In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import httpx
import roboflow


# Set this to the run printed by the training wizard.
checkpoint_path = Path("runs/<rfdetr-run>/checkpoint_best_ema.pth")

roboflow.login()
rf = roboflow.Roboflow()

In [ ]:
def post_to_roboflow(url, payload, operation):
    try:
        response = httpx.post(
            url, params={"api_key": rf.api_key}, json=payload, timeout=30
        )
        response.raise_for_status()
        return response
    except httpx.HTTPStatusError as exc:
        print(f"{operation} failed (HTTP {exc.response.status_code})")
    except httpx.RequestError as exc:
        print(f"{operation} failed ({type(exc).__name__})")
    return None


def exact_filename_lookup(filename, split):
    response = post_to_roboflow(
        "https://api.roboflow.com/agrowizard/search/v1",
        {
            "query": f"filename:{filename} AND split:{split}",
            "pageSize": 10,
            "fields": ["id", "filename"],
        },
        f"Searching for {filename} in {split}",
    )
    if response is None:
        return None
    try:
        results = response.json()["results"]
    except (ValueError, KeyError, TypeError):
        print(f"Invalid search response for {filename} in {split}")
        return None
    return [item for item in results if item.get("filename") == filename]


def original_filename(overlay_name, direction):
    suffix = "_fp.jpg" if direction == "false_positives" else "_fn.jpg"
    exported = overlay_name.removesuffix(suffix)
    base, separator, _ = exported.partition(".rf.")
    if separator:
        name, separator, extension = base.rpartition("_")
        if separator and extension.lower() in {"jpg", "jpeg", "png", "avif", "webp"}:
            return f"{name}.{extension}"
    return exported

In [ ]:
def add_tag(image_id, tag):
    return post_to_roboflow(
        f"https://api.roboflow.com/agrowizard/stick-detection/images/{image_id}/tags",
        {"operation": "add", "tags": [tag]},
        f"Adding {tag} to image {image_id}",
    ) is not None

In [ ]:
if not checkpoint_path.is_file():
    raise FileNotFoundError(f"Set checkpoint_path to an existing checkpoint: {checkpoint_path}")

tagged = failed_tags = failed_searches = unmatched = 0
with ThreadPoolExecutor(max_workers=10) as executor:
    for split in ("valid", "test", "train"):
        prefix = f"{checkpoint_path.stem}_{split}_evaluation"
        summaries = list(checkpoint_path.parent.glob(f"{prefix}/summary.json"))
        summaries += list(checkpoint_path.parent.glob(f"{prefix}-*/summary.json"))
        if not summaries:
            continue
        report = max(summaries, key=lambda path: path.stat().st_mtime_ns).parent
        for direction in ("false_positives", "false_negatives"):
            suffix = "_fp.jpg" if direction == "false_positives" else "_fn.jpg"
            for category in ("sticks", "poles", "trees"):
                directory = report / direction / category
                files = {
                    original_filename(path.name, direction)
                    for path in directory.rglob(f"*{suffix}")
                    if path.is_file()
                }
                if not files:
                    continue

                matches = list(executor.map(
                    lambda filename: exact_filename_lookup(filename, split), sorted(files)
                ))
                failed_searches += sum(result is None for result in matches)
                unmatched += sum(result == [] for result in matches)
                image_ids = {image["id"] for result in matches if result for image in result}
                tag = f"{category}-{direction.replace('_', '-')}"
                results = list(executor.map(lambda image_id: add_tag(image_id, tag), sorted(image_ids)))
                successes = sum(results)
                tagged += successes
                failed_tags += len(results) - successes
                print(f"{split}: applied {successes}/{len(results)} {tag} tags")

print(
    f"Applied {tagged} tags. Failed tags: {failed_tags}. "
    f"Failed searches: {failed_searches}. Unmatched files: {unmatched}."
)
if failed_tags or failed_searches:
    raise RuntimeError("Some Roboflow requests failed.")